# Research · Dehazing — Learned reference: AOD-Net vs the tuned DCP (NB 3 / 3)

**Group 01 · Dept. of CSE, East West University** — Md. Asif Hossain (2022-3-60-007) · Nabil Subhan (2022-3-60-063) · K M Nudar (2022-3-60-234)

**Course:** CSE 348 / 438 — Digital Image Processing · **Research project** (Idea Bank: Filtering & Restoration → *Image dehazing using the dark channel prior*)

**Purpose.** Position the classical pipeline against a learned dehazer on the *identical* test images and metrics, and test
the density interaction (H3) between them. **AOD-Net** (Li et al., ICCV 2017) is used because it is tiny (~1.8 k parameters),
trains in minutes, and re-parameterises the same haze model DCP inverts (`J = K·I − K + b`).

**Weights, in order of preference:** (1) a pretrained AOD-Net state-dict attached as a dataset (any `*.pth` / `*.pt` with
five conv layers); (2) training here on RESIDE-ITS/OTS pairs if a *training* set is attached; (3) training here on
synthetic haze rendered from the **tune-split ground truths only** (test images are never seen). The route that ran is
printed and written to the results file — report it. A GPU makes (2)/(3) faster but is not required.

Attach the same evaluation datasets as NB 1/2 **and** NB 2's output (for `dehaze_results.json` → the tuned DCP configuration).


In [ ]:
# ===== Dehazing research library (identical cell in NB1 / NB2 / NB3 — edit in one, copy to all) =====
import os, re, json, time, math, warnings
from pathlib import Path
import numpy as np, pandas as pd, cv2
from PIL import Image
import matplotlib.pyplot as plt
from scipy import sparse, stats
from scipy.sparse.linalg import cg as _cg
from skimage.metrics import peak_signal_noise_ratio as _psnr, structural_similarity as _ssim
from skimage.color import rgb2lab, deltaE_ciede2000
warnings.filterwarnings("ignore")

SEED = 42
IS_KAGGLE = Path("/kaggle").exists()
INPUT_ROOT = Path(os.environ.get("DEHAZE_INPUT_ROOT", "/kaggle/input"))
WORK = Path(os.environ.get("DEHAZE_WORK", "/kaggle/working" if IS_KAGGLE else "dehaze_work")); WORK.mkdir(parents=True, exist_ok=True)
FAST = os.environ.get("DEHAZE_FAST", "0") == "1"          # quick smoke-run: fewer images / bootstraps

CONFIG = dict(
    max_side=512,               # images resized so the longer side <= max_side (metrics computed at this size)
    max_per_dataset=60 if FAST else 500,
    tune_frac=0.30, min_tune=5, # scene-grouped tune/test split (tune never exceeds half the scenes); knobs chosen on TUNE, reported on TEST
    # DCP baseline (He, Sun & Tang 2009/2011 defaults)
    patch=15, omega=0.95, t0=0.10, A_method="dcp_top", refine="guided", gf_r=40, gf_eps=1e-3,
    # bright / sky-like region mask (HSV): high value, low saturation
    bright_v=0.75, bright_s=0.25, min_bright_px=200,
    n_boot=300 if FAST else 1000,
    matting_max=6 if FAST else 30, matting_side=320, matting_lambda=1e-4, matting_eps=1e-7,
)
print(f"env: {'Kaggle' if IS_KAGGLE else 'local'} | input {INPUT_ROOT} | work {WORK} | FAST={FAST}")

# ---------- figure style: validated palette (dataviz validator, light surface #fcfcfb) ----------
C_BLUE, C_ORANGE, C_AQUA, C_YELLOW, C_MAGENTA, C_VIOLET, C_RED = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#4a3aa7", "#e34948"
SERIES = [C_BLUE, C_ORANGE, C_AQUA, C_YELLOW, C_MAGENTA]        # categorical, fixed order (adjacent-pair validated)
RAMP3 = ["#86b6ef", "#2a78d6", "#104281"]                        # ordinal light -> medium -> dense (single-hue, validated)
INK, INK2, MUTED, GRIDC, RULE, SURF = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "font.size": 9.5, "font.family": "DejaVu Sans",
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": RULE, "axes.labelcolor": INK2,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True, "grid.color": GRIDC, "grid.linewidth": 0.6,
    "axes.axisbelow": True, "axes.titleweight": "bold", "axes.titlecolor": INK, "axes.titlesize": 10,
    "legend.frameon": False, "figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF})
METHOD_COLORS = {"hazy input": MUTED, "CLAHE": C_YELLOW, "DCP baseline": C_BLUE, "DCP tuned": C_VIOLET, "AOD-Net": C_ORANGE}
import matplotlib as _mpl, scipy as _scipy, skimage as _skimage
BOXLBL = "tick_labels" if tuple(int(x) for x in _mpl.__version__.split(".")[:2]) >= (3, 9) else "labels"   # boxplot kwarg renamed in 3.9
print(f"numpy {np.__version__} | opencv {cv2.__version__} | scipy {_scipy.__version__} | scikit-image {_skimage.__version__} | matplotlib {_mpl.__version__} | pandas {pd.__version__}")

def savefig(name):
    """Save PNG (for notebooks/README) and PDF (for the IEEE report) with one call."""
    plt.savefig(WORK / f"{name}.png", bbox_inches="tight"); plt.savefig(WORK / f"{name}.pdf", bbox_inches="tight")

def table_png(df, name, title=None, fmt="{:.3f}", highlight_max=(), highlight_min=()):
    """Render a DataFrame as a publication-style table image (also written as CSV)."""
    df.to_csv(WORK / f"{name}.csv")
    cells = [[(fmt.format(v) if isinstance(v, (float, np.floating)) and not (isinstance(v, float) and math.isnan(v)) else str(v)) for v in row] for row in df.values]
    fig, ax = plt.subplots(figsize=(min(14, 1.15 + 1.35 * len(df.columns)), 0.5 + 0.34 * (len(df) + 1)))
    ax.axis("off")
    tbl = ax.table(cellText=cells, colLabels=list(df.columns), rowLabels=[str(i) for i in df.index], loc="center", cellLoc="center")
    tbl.auto_set_font_size(False); tbl.set_fontsize(8.5); tbl.scale(1, 1.35)
    for (r, c), cell in tbl.get_celld().items():
        cell.set_edgecolor(GRIDC); cell.set_linewidth(0.6)
        if r == 0: cell.set_facecolor("#e9eef5"); cell.set_text_props(weight="bold", color=INK)
        elif c == -1: cell.set_text_props(color=INK2)
    for col in highlight_max:
        if col in df.columns:
            j = list(df.columns).index(col); i = int(np.nanargmax(df[col].values.astype(float)))
            tbl[(i + 1, j)].set_facecolor("#dcefe5"); tbl[(i + 1, j)].set_text_props(weight="bold")
    for col in highlight_min:
        if col in df.columns:
            j = list(df.columns).index(col); i = int(np.nanargmin(df[col].values.astype(float)))
            tbl[(i + 1, j)].set_facecolor("#dcefe5"); tbl[(i + 1, j)].set_text_props(weight="bold")
    if title: ax.set_title(title, fontweight="bold", pad=8, color=INK)
    plt.savefig(WORK / f"{name}.png", bbox_inches="tight"); plt.show()

# ---------- Dark Channel Prior: every stage is one named classical operator ----------
def dark_channel(img, patch=15):
    """J_dark(x) = min_c min_{y in Omega(x)} J_c(y): per-pixel channel minimum, then a (patch x patch) grey erosion."""
    mn = img.min(axis=2)
    k = cv2.getStructuringElement(cv2.MORPH_RECT, (int(patch), int(patch)))
    return cv2.erode(mn, k)

def estimate_A(img, dark, method="dcp_top"):
    """Atmospheric light A (3-vector in [0,1]).
    brightest     : brightest input pixel (naive; sky/white objects hijack it)
    dcp_top       : He et al. — among the 0.1% brightest dark-channel pixels, the input pixel with highest intensity
    dcp_top_mean  : mean of those 0.1% candidates (robust to single outliers)
    quadtree      : Kim et al. 2013 — recursively keep the quadrant maximising mean - std, then brightest pixel there
    """
    flat = img.reshape(-1, 3)
    if method == "brightest":
        return flat[flat.sum(1).argmax()].copy()
    if method == "quadtree":
        reg = img
        while reg.shape[0] * reg.shape[1] > 400 and min(reg.shape[:2]) >= 4:
            H, W = reg.shape[:2]; hh, ww = H // 2, W // 2
            quads = [reg[:hh, :ww], reg[:hh, ww:], reg[hh:, :ww], reg[hh:, ww:]]
            score = [q.reshape(-1, 3).mean() - q.reshape(-1, 3).std() for q in quads]
            reg = quads[int(np.argmax(score))]
        fr = reg.reshape(-1, 3); return fr[fr.sum(1).argmax()].copy()
    n = max(1, int(0.001 * dark.size))
    idx = np.argpartition(dark.ravel(), -n)[-n:]
    cand = flat[idx]
    if method == "dcp_top_mean":
        return cand.mean(0)
    return cand[cand.sum(1).argmax()].copy()                      # dcp_top

def transmission_raw(img, A, omega=0.95, patch=15):
    """t~(x) = 1 - omega * dark_channel(I / A). omega < 1 keeps a trace of haze for depth perception."""
    return 1.0 - omega * dark_channel(img / np.maximum(A, 1e-6)[None, None, :], patch)

def guided_filter(guide, src, r=40, eps=1e-3):
    """He, Sun & Tang 2010 (grey guide). Local linear model q = a*I + b in (2r+1)^2 windows; O(N) via box filters."""
    k = (2 * int(r) + 1, 2 * int(r) + 1)
    box = lambda x: cv2.boxFilter(x, cv2.CV_64F, k, normalize=True, borderType=cv2.BORDER_REFLECT)
    mI, mp = box(guide), box(src)
    cov = box(guide * src) - mI * mp
    var = box(guide * guide) - mI * mI
    a = cov / (var + eps); b = mp - a * mI
    return box(a) * guide + box(b)

def matting_laplacian(img, eps=1e-7):
    """Levin et al. 2008 closed-form matting Laplacian (3x3 windows) as a sparse matrix — the refinement He et al. 2009 used."""
    from numpy.lib.stride_tricks import sliding_window_view
    h, w, c = img.shape; n = h * w; ws = 9
    idx = np.arange(n).reshape(h, w)
    win_idx = sliding_window_view(idx, (3, 3)).reshape(-1, ws)                                  # (m, 9)
    win_I = sliding_window_view(img, (3, 3), axis=(0, 1)).reshape(-1, c, ws).transpose(0, 2, 1)   # (m, 9, 3)
    mu = win_I.mean(1, keepdims=True); X = win_I - mu
    cov = np.einsum("mki,mkj->mij", X, X) / ws
    inv = np.linalg.inv(cov + (eps / ws) * np.eye(c)[None])
    vals = (1.0 + np.einsum("mki,mij,mlj->mkl", X, inv, X)) / ws                                # (m, 9, 9)
    rows = np.repeat(win_idx, ws, axis=1).ravel(); cols = np.tile(win_idx, (1, ws)).ravel()
    Lw = sparse.coo_matrix((vals.ravel(), (rows, cols)), shape=(n, n)).tocsr()
    return sparse.diags(np.asarray(Lw.sum(1)).ravel()) - Lw

def matting_refine(img, t_raw, lam=1e-4, eps=1e-7, side=320):
    """Solve (L + lam*I) t = lam * t~ at a reduced resolution (conjugate gradients), then upsample."""
    h, w = t_raw.shape; s = min(1.0, side / max(h, w))
    if s < 1.0:
        im = cv2.resize(img, (max(8, int(w * s)), max(8, int(h * s))), interpolation=cv2.INTER_AREA)
        tr = cv2.resize(t_raw, (im.shape[1], im.shape[0]), interpolation=cv2.INTER_AREA)
    else:
        im, tr = img, t_raw
    L = matting_laplacian(im, eps); n = L.shape[0]
    Aop = L + lam * sparse.identity(n, format="csr")
    try: t, _ = _cg(Aop, lam * tr.ravel(), x0=tr.ravel(), rtol=1e-4, maxiter=800)
    except TypeError: t, _ = _cg(Aop, lam * tr.ravel(), x0=tr.ravel(), tol=1e-4, maxiter=800)
    t = t.reshape(tr.shape)
    if s < 1.0: t = cv2.resize(t, (w, h), interpolation=cv2.INTER_LINEAR)
    return np.clip(t, 0, 1)

def recover(img, A, t, t0=0.10):
    """Invert the haze model I = J t + A (1 - t):  J = (I - A) / max(t, t0) + A. The floor t0 bounds noise amplification."""
    tt = np.clip(t, t0, 1.0)[..., None]
    return np.clip((img - A[None, None, :]) / tt + A[None, None, :], 0.0, 1.0)

def to_gray(img): return cv2.cvtColor((img * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float64) / 255.0

def dehaze(img, patch=15, omega=0.95, t0=0.10, A_method="dcp_top", refine="guided", gf_r=40, gf_eps=1e-3, **kw):
    """Full DCP pipeline; returns every intermediate so any stage can be inspected or ablated."""
    dark = dark_channel(img, patch)
    A = estimate_A(img, dark, A_method)
    t_raw = transmission_raw(img, A, omega, patch)
    if refine == "guided":
        t = guided_filter(to_gray(img), t_raw, gf_r, gf_eps)
    elif refine == "matting":
        t = matting_refine(img, t_raw, CONFIG["matting_lambda"], CONFIG["matting_eps"], CONFIG["matting_side"])
    else:
        t = t_raw
    J = recover(img, A, t, t0)
    return {"dehazed": J, "dark": dark, "A": A, "t_raw": t_raw, "t": np.clip(t, 0, 1)}

def clahe_enhance(img, clip=2.0, tile=8):
    """Classical contrast-enhancement control (no haze model): CLAHE on the L channel of Lab."""
    lab = cv2.cvtColor((img * 255).astype(np.uint8), cv2.COLOR_RGB2LAB)
    lab[..., 0] = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tile, tile)).apply(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB).astype(np.float64) / 255.0

# ---------- metrics: full-reference (PSNR, SSIM, CIEDE2000) + no-reference (Hautière e, r; DCP density) ----------
def psnr(out, gt): return float(_psnr(gt, out, data_range=1.0))
def ssim(out, gt): return float(_ssim(gt.astype(np.float32), out.astype(np.float32), data_range=1.0, channel_axis=2))
def ciede(out, gt, side=256):
    """Mean CIEDE2000 colour difference in CIELAB; evaluated on an area-downsampled copy (<= side px) — a spatial mean is insensitive to this, 4x faster."""
    h, w = gt.shape[:2]; f = side / max(h, w)
    if f < 1.0:
        gt = cv2.resize(gt, (max(8, int(w * f)), max(8, int(h * f))), interpolation=cv2.INTER_AREA); out = cv2.resize(out, (gt.shape[1], gt.shape[0]), interpolation=cv2.INTER_AREA)
    return float(deltaE_ciede2000(rgb2lab(gt.astype(np.float32)), rgb2lab(out.astype(np.float32))).mean())

def hautiere(hazy, out):
    """Hautière et al. 2008 blind contrast descriptors: e = rate of new visible edges, r = geometric-mean gradient ratio on visible edges."""
    g0, g1 = to_gray(hazy), to_gray(out)
    e0 = cv2.Canny((g0 * 255).astype(np.uint8), 50, 150) > 0
    e1 = cv2.Canny((g1 * 255).astype(np.uint8), 50, 150) > 0
    n0, n1 = int(e0.sum()), int(e1.sum())
    e = (n1 - n0) / max(n0, 1)
    grad = lambda g: np.hypot(cv2.Sobel(g, cv2.CV_64F, 1, 0, ksize=3), cv2.Sobel(g, cv2.CV_64F, 0, 1, ksize=3))
    G0, G1 = grad(g0), grad(g1)
    m = e1 & (G0 > 1e-3) & (G1 > 1e-3)
    r = float(np.exp(np.mean(np.log(G1[m] / G0[m])))) if m.sum() > 10 else float("nan")
    return e, r

def haze_density(img, patch=15):
    """No-reference haze-density proxy: mean dark channel of the input (DCP's own statistic; 0 = clear, 1 = opaque)."""
    return float(dark_channel(img, patch).mean())

def bright_mask(img, v=None, s=None):
    """Sky-like / bright-region mask (HSV value high, saturation low) — where the dark-channel prior is violated."""
    hsv = cv2.cvtColor((img * 255).astype(np.uint8), cv2.COLOR_RGB2HSV).astype(np.float64)
    return (hsv[..., 2] / 255.0 > (v or CONFIG["bright_v"])) & (hsv[..., 1] / 255.0 < (s or CONFIG["bright_s"]))

def score_all(out, gt, hazy):
    e, r = hautiere(hazy, out)
    return dict(psnr=psnr(out, gt), ssim=ssim(out, gt), ciede=ciede(out, gt), e=e, r=r)

# ---------- statistics: paired bootstrap CI + Wilcoxon signed-rank on the SAME images ----------
def paired_stats(a, b, n_boot=None, seed=0):
    """Paired difference a - b per image: mean, 95% bootstrap CI of the mean, Wilcoxon p."""
    a, b = np.asarray(a, float), np.asarray(b, float); ok = ~(np.isnan(a) | np.isnan(b)); d = (a - b)[ok]
    if len(d) == 0: return dict(mean=np.nan, lo=np.nan, hi=np.nan, p=np.nan, n=0)
    rng = np.random.default_rng(seed); nb = n_boot or CONFIG["n_boot"]
    boots = rng.choice(d, (nb, len(d)), replace=True).mean(1)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    p = float(stats.wilcoxon(d).pvalue) if np.any(d != 0) and len(d) >= 6 else float("nan")
    return dict(mean=float(d.mean()), lo=float(lo), hi=float(hi), p=p, n=int(len(d)))

def mean_ci(x, n_boot=None, seed=0):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if len(x) == 0: return (np.nan, np.nan, np.nan)
    rng = np.random.default_rng(seed); nb = n_boot or CONFIG["n_boot"]
    boots = rng.choice(x, (nb, len(x)), replace=True).mean(1)
    return float(x.mean()), float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))

# ---------- data: discover paired (hazy, clear) benchmarks under /kaggle/input, else synthesise ----------
IMG_EXT = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}
HAZY_DIR = {"hazy", "haze", "hazy_images", "hazy_image", "haze_images", "input", "inputs", "hazed", "synthetic", "hazy_imgs"}
GT_DIR = {"gt", "clear", "clean", "clear_images", "clean_images", "target", "targets", "ground_truth", "groundtruth", "original", "reference", "sharp", "gt_images", "clear_imgs"}

def _dataset_name(path):
    """-> (name, is_real, is_training_set). Training sets (RESIDE ITS/OTS, anything named train) are excluded from evaluation."""
    p = "/" + str(path).lower().replace("\\", "/") + "/"
    tok = lambda *ks: any(re.search(r"[/_\-\s#(](" + k + r")[/_\-\s)]", p) for k in ks)
    if "dense" in p: return "Dense-Haze", True, False
    if tok("nh-haze", "nhhaze", "nh_haze", "nh"): return "NH-HAZE", True, False
    if tok("o-haze", "o-hazy", "ohaze", "o_haze", "o-haz", "ohazy"): return "O-HAZE", True, False
    if tok("i-haze", "i-hazy", "ihaze", "i_haze", "i-haz", "ihazy"): return "I-HAZE", True, False
    if tok("its", "ots", "train", "training"): return ("RESIDE-ITS-train" if tok("its") else "RESIDE-OTS-train" if tok("ots") else "train"), False, True
    if tok("sots", "reside", "nyuhaze500", "nyu"):
        if "indoor" in p or tok("nyuhaze500", "nyu"): return "SOTS-indoor", False, False
        if "outdoor" in p: return "SOTS-outdoor", False, False
        return "RESIDE", False, False
    return Path(path).name or "paired", False, False

def _key(stem):
    k = stem.split("_")[0].split("-")[0]
    d = "".join(ch for ch in k if ch.isdigit())
    return d.lstrip("0") or d or k.lower()

def _images(d): return sorted(p for p in d.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXT)

def discover_pairs(root=INPUT_ROOT):
    """Find every (hazy, clear) pair under root. Handles hazy/ + GT/ sibling folders (RESIDE-SOTS, O/I/Dense/NH-HAZE
    Kaggle mirrors) and flat folders with *_hazy / *_GT suffixes. Pairs by the leading numeric key of the file stem."""
    root = Path(root); recs = []
    if not root.exists(): return recs
    seen = set()
    for d in sorted(p for p in root.rglob("*") if p.is_dir()):
        nm = d.name.lower()
        if nm in HAZY_DIR and _images(d):
            gt = next((s for s in d.parent.iterdir() if s.is_dir() and s.name.lower() in GT_DIR and _images(s)), None)
            if gt is None: continue
            gmap = {}
            for g in _images(gt): gmap.setdefault(_key(g.stem), g)
            name, real, train = _dataset_name(d.parent)
            for hz in _images(d):
                g = gmap.get(_key(hz.stem))
                if g is None: continue
                if (name, hz.name) in seen: continue
                seen.add((name, hz.name))
                recs.append(dict(dataset=name, real=real, train=train, key=f"{name}:{_key(hz.stem)}", name=hz.stem, hazy=str(hz), gt=str(g)))
        else:                                                    # flat layout: 01_hazy.png + 01_GT.png in one folder
            ims = _images(d)
            hz_ims = [p for p in ims if re.search(r"(hazy|haze)", p.stem, re.I)]
            gt_ims = {_key(p.stem): p for p in ims if re.search(r"(gt|clear|clean)", p.stem, re.I) and not re.search(r"(hazy|haze)", p.stem, re.I)}
            if hz_ims and gt_ims:
                name, real, train = _dataset_name(d)
                for hz in hz_ims:
                    g = gt_ims.get(_key(hz.stem))
                    if g is None or (name, hz.name) in seen: continue
                    seen.add((name, hz.name))
                    recs.append(dict(dataset=name, real=real, train=train, key=f"{name}:{_key(hz.stem)}", name=hz.stem, hazy=str(hz), gt=str(g)))
    return recs

def load_rgb(src, max_side=None):
    """Path or array -> float64 RGB in [0,1], longer side <= max_side (aspect preserved)."""
    ms = max_side or CONFIG["max_side"]
    if isinstance(src, np.ndarray):
        im = Image.fromarray((src * 255).astype(np.uint8) if src.dtype != np.uint8 else src)
    else:
        im = Image.open(src); im = im.convert("RGB")
    im = im.convert("RGB")
    if max(im.size) > ms: im.thumbnail((ms, ms), Image.LANCZOS)
    return np.asarray(im, np.float64) / 255.0

def load_pair(rec):
    hz, gt = load_rgb(rec["hazy"]), load_rgb(rec["gt"])
    if hz.shape != gt.shape: gt = cv2.resize(gt, (hz.shape[1], hz.shape[0]), interpolation=cv2.INTER_AREA)
    return hz, gt

# synthetic fallback — keeps every notebook runnable with no dataset attached (results are then labelled 'synthetic')
def _depth_map(h, w, kind, rng):
    yy, xx = np.mgrid[0:h, 0:w]
    if kind == "vertical": d = 1.0 - yy / max(h - 1, 1)                                   # far at the top (sky-like)
    elif kind == "radial": d = np.sqrt(((xx - w / 2) / w) ** 2 + ((yy - h / 2) / h) ** 2) * 1.6
    else:
        d = cv2.GaussianBlur(rng.random((h, w)), (0, 0), max(h, w) / 8); d = (d - d.min()) / (np.ptp(d) + 1e-9)
    return 0.15 + 0.85 * np.clip(d, 0, 1)

def synth_haze(clear, beta, A, depth):
    t = np.exp(-beta * depth)[..., None]
    return np.clip(clear * t + np.asarray(A)[None, None, :] * (1 - t), 0, 1), t[..., 0]

def synthetic_records():
    from skimage import data as skd
    rng = np.random.default_rng(SEED); clears = []
    for nm in ["astronaut", "coffee", "chelsea", "rocket", "immunohistochemistry", "retina", "colorwheel", "logo", "camera", "brick", "grass"]:
        try:
            im = getattr(skd, nm)()
            if im.ndim == 2: im = np.stack([im] * 3, -1)
            if im.dtype != np.uint8: im = (255 * (im - im.min()) / (np.ptp(im) + 1e-9)).astype(np.uint8)
            clears.append((nm, load_rgb(im[..., :3])))
        except Exception: pass
    h = w = 384; yy, xx = np.mgrid[0:h, 0:w]                                            # procedural 'sky over field' scene
    sky = np.stack([0.62 + 0.3 * (1 - yy / h), 0.72 + 0.22 * (1 - yy / h), 0.9 + 0.08 * (1 - yy / h)], -1)
    field = np.stack([0.25 + 0.15 * rng.random((h, w)), 0.45 + 0.2 * rng.random((h, w)), 0.15 + 0.1 * rng.random((h, w))], -1)
    field = cv2.GaussianBlur(field, (0, 0), 1.2); scene = np.where((yy < h * 0.45)[..., None], sky, field)
    tree = ((xx - w * 0.7) ** 2 / 900 + (yy - h * 0.5) ** 2 / 3600) < 1; scene[tree] = [0.1, 0.25, 0.08]
    clears.append(("skyfield", np.clip(scene, 0, 1)))
    recs = []
    for nm, c in clears:
        for beta in (0.8, 1.6, 2.6):
            for A in ([0.85, 0.87, 0.90], [0.75, 0.77, 0.80]):
                kind = ["vertical", "radial", "smooth"][len(recs) % 3]
                depth = _depth_map(c.shape[0], c.shape[1], kind, rng)
                hz, _ = synth_haze(c, beta, A, depth)
                recs.append(dict(dataset="synthetic", real=False, train=False, key=f"synthetic:{nm}", name=f"{nm}_b{beta}_A{A[0]}",
                                 hazy=hz.astype(np.float32), gt=c.astype(np.float32), beta=beta, A_true=A))
    return recs

def build_records():
    """Evaluation records: every discovered benchmark pair except training sets, capped per dataset (seeded)."""
    allrecs = discover_pairs(INPUT_ROOT)
    tr = sorted({r["dataset"] for r in allrecs if r.get("train")})
    if tr: print("training-only sets found (excluded from evaluation, available to NB 3):", {t: sum(r["dataset"] == t for r in allrecs) for t in tr})
    recs = [r for r in allrecs if not r.get("train")]
    if recs:
        rng = np.random.default_rng(SEED); out = []
        for ds in sorted({r["dataset"] for r in recs}):
            sub = [r for r in recs if r["dataset"] == ds]
            if len(sub) > CONFIG["max_per_dataset"]:
                sub = [sub[i] for i in sorted(rng.choice(len(sub), CONFIG["max_per_dataset"], replace=False))]
            out += sub
        source = "benchmarks under " + str(INPUT_ROOT)
        return out, source
    return synthetic_records(), "synthetic-haze fallback (no paired dataset attached)"

def split_tune_test(recs, frac=None, seed=SEED):
    """Scene-grouped split: every hazy version of one clear scene lands in the same split (SOTS has 10 hazy/scene)."""
    frac = frac or CONFIG["tune_frac"]; rng = np.random.default_rng(seed)
    for ds in sorted({r["dataset"] for r in recs}):
        keys = sorted({r["key"] for r in recs if r["dataset"] == ds}); rng.shuffle(keys)
        n_t = min(len(keys) // 2, max(CONFIG["min_tune"], int(round(frac * len(keys))))) if len(keys) > 1 else 0
        tune = set(keys[:n_t])
        for r in recs:
            if r["dataset"] == ds: r["split"] = "tune" if r["key"] in tune else "test"
    return recs

def get_pair(rec):
    return (rec["hazy"].astype(np.float64), rec["gt"].astype(np.float64)) if isinstance(rec["hazy"], np.ndarray) else load_pair(rec)

def fingerprint(recs):
    import hashlib
    return hashlib.sha256("|".join(sorted(f"{r['dataset']}/{r['name']}/{r['split']}" for r in recs)).encode()).hexdigest()[:12]

def tex_macros(d, path):
    """Write \\newcommand macros (letters only) so the LaTeX report pulls every number from the run."""
    def clean(k): return re.sub(r"[^A-Za-z]", "", k)
    with open(path, "w", encoding="utf-8") as f:
        for k, v in d.items():
            f.write(f"\\newcommand{{\\{clean(k)}}}{{{v}}}\n")


## 1. Data (same split), tuned DCP configuration from NB 2


In [ ]:
# ===== paired data: benchmarks attached under /kaggle/input, else the synthetic fallback =====
records, SOURCE = build_records()
records = split_tune_test(records)
FP = fingerprint(records)
inv = (pd.DataFrame([{"dataset": r["dataset"], "real": r["real"], "split": r["split"]} for r in records])
         .groupby(["dataset", "real", "split"]).size().unstack("split").fillna(0).astype(int))
inv["total"] = inv.sum(1)
print(f"source: {SOURCE}\nimages: {len(records)} | split fingerprint {FP} (scene-grouped tune/test, seed {SEED})")
print(inv.to_string())

stat_rows = []
for r in records:
    hz, g = get_pair(r); stat_rows.append({"name": r["name"], "density": haze_density(hz)})
S = pd.DataFrame(stat_rows); q1, q2 = S.density.quantile([1/3, 2/3]).values
S["stratum"] = np.where(S.density <= q1, "light", np.where(S.density <= q2, "medium", "dense")); STRATA = ["light", "medium", "dense"]
for r, st in zip(records, S.stratum): r["stratum"] = st
BASE = dict(patch=15, omega=0.95, t0=0.10, A_method="dcp_top", refine="guided", gf_r=40, gf_eps=1e-3)
cands = list(INPUT_ROOT.rglob("dehaze_results.json")) + list(WORK.glob("dehaze_results.json"))
if cands:
    prev = json.load(open(cands[0])); tuned = prev["tuned_cfg"]
    if prev.get("fingerprint") != FP: print("WARNING: NB 2 ran on a different image set/split (fingerprint mismatch) — attach the matching NB 2 output")
    print("tuned DCP config from", cands[0], "->", tuned)
else:
    tuned = dict(BASE); print("NB 2 output not attached -> using the He et al. baseline as 'tuned' (attach NB 2's output for the real comparison)")


## 2. AOD-Net — load pretrained weights, or train here


In [ ]:
try:
    import torch, torch.nn as nn, torch.nn.functional as Fnn
    TORCH = True; device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); print("torch", torch.__version__, "| device", device)
except Exception as e:
    TORCH = False; print("torch unavailable ->", e)

if TORCH:
    class AODNet(nn.Module):
        """Li et al. 2017: K(x) estimated by 5 convs with multi-scale concatenation; J = K*I - K + b."""
        def __init__(self, b=1.0):
            super().__init__(); self.b = b
            self.e_conv1 = nn.Conv2d(3, 3, 1); self.e_conv2 = nn.Conv2d(3, 3, 3, padding=1); self.e_conv3 = nn.Conv2d(6, 3, 5, padding=2)
            self.e_conv4 = nn.Conv2d(6, 3, 7, padding=3); self.e_conv5 = nn.Conv2d(12, 3, 3, padding=1)
        def forward(self, x):
            x1 = Fnn.relu(self.e_conv1(x)); x2 = Fnn.relu(self.e_conv2(x1)); x3 = Fnn.relu(self.e_conv3(torch.cat([x1, x2], 1)))
            x4 = Fnn.relu(self.e_conv4(torch.cat([x2, x3], 1))); k = Fnn.relu(self.e_conv5(torch.cat([x1, x2, x3, x4], 1)))
            return Fnn.relu(k * x - k + self.b)
    net = AODNet().to(device); ROUTE = None
    # (1) pretrained state-dict attached?
    for wp in sorted(list(INPUT_ROOT.rglob("*.pth")) + list(INPUT_ROOT.rglob("*.pt"))):
        try:
            sd = torch.load(wp, map_location="cpu"); sd = sd.get("state_dict", sd) if isinstance(sd, dict) else sd
            sd = {k.replace("module.", ""): v for k, v in sd.items()}
            mine = net.state_dict()
            if set(sd) == set(mine): net.load_state_dict(sd)
            else:                                            # map by order: 5 conv weight/bias pairs
                ws = [v for k, v in sd.items() if v.ndim == 4]; bs = [v for k, v in sd.items() if v.ndim == 1]
                assert len(ws) == 5 and len(bs) == 5 and all(w.shape == m.shape for w, m in zip(ws, [v for k, v in mine.items() if v.ndim == 4]))
                net.load_state_dict({k: (ws if v.ndim == 4 else bs)[i // 2] for i, (k, v) in enumerate(mine.items())})
            ROUTE = f"pretrained weights: {wp.name}"; break
        except Exception as e: print("  skip", wp.name, "->", type(e).__name__)
    # (2) a RESIDE training set attached? (ITS / OTS folders, not SOTS)
    train_pairs = [] if ROUTE else [p for p in discover_pairs(INPUT_ROOT) if p.get("train")]
    if not ROUTE and len(train_pairs) >= 200:
        rng = np.random.default_rng(SEED); train_pairs = [train_pairs[i] for i in rng.choice(len(train_pairs), min(3000, len(train_pairs)), replace=False)]
        ROUTE = f"trained here on {len(train_pairs)} RESIDE training pairs"
        def sample_batch(bs=8, crop=240):
            pairs = [load_pair(train_pairs[i]) for i in rng.choice(len(train_pairs), bs)]
            c = min([crop] + [min(hz.shape[:2]) for hz, _ in pairs])            # one crop size per batch (images may be small)
            xs, ys = [], []
            for hz, g in pairs:
                h, w = hz.shape[:2]; y0, x0 = rng.integers(0, h - c + 1), rng.integers(0, w - c + 1)
                xs.append(hz[y0:y0+c, x0:x0+c]); ys.append(g[y0:y0+c, x0:x0+c])
            return xs, ys
    elif not ROUTE:                                          # (3) synthetic haze from TUNE-split ground truths only
        tune_gts = [get_pair(r)[1] for r in records if r["split"] == "tune"]
        seen_keys = {r["key"] for r in records if r["split"] == "tune"}
        ROUTE = f"trained here on synthetic haze from {len(tune_gts)} tune-split GT images (test scenes unseen)"
        rng = np.random.default_rng(SEED)
        def sample_batch(bs=8, crop=240):
            xs, ys = [], []
            gts = [tune_gts[i] for i in rng.choice(len(tune_gts), bs)]
            c = min([crop] + [min(g.shape[:2]) for g in gts])                    # one crop size per batch
            for g in gts:
                h, w = g.shape[:2]; y0, x0 = rng.integers(0, h - c + 1), rng.integers(0, w - c + 1); gc = g[y0:y0+c, x0:x0+c]
                depth = _depth_map(c, c, ["vertical", "radial", "smooth"][rng.integers(3)], rng)
                A = float(rng.uniform(0.7, 1.0)); A = [A, A + rng.uniform(-.03, .03), A + rng.uniform(-.03, .03)]
                hz, _ = synth_haze(gc, float(rng.uniform(0.5, 3.0)), np.clip(A, 0, 1), depth); xs.append(hz); ys.append(gc)
            return xs, ys
    print("AOD-Net route:", ROUTE)
    if "trained here" in ROUTE:
        EPOCHS = int(os.environ.get("DEHAZE_AOD_EPOCHS", 3 if FAST else (40 if device.type == "cuda" else 12))); STEPS = 40 if FAST else 150
        opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-4); hist = []; t0_ = time.time()
        for ep in range(1, EPOCHS + 1):
            net.train(); losses = []
            for _ in range(STEPS):
                xs, ys = sample_batch()
                x = torch.tensor(np.stack(xs), dtype=torch.float32).permute(0, 3, 1, 2).to(device); y = torch.tensor(np.stack(ys), dtype=torch.float32).permute(0, 3, 1, 2).to(device)
                loss = Fnn.mse_loss(net(x), y); opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())
            hist.append(np.mean(losses)); print(f"  epoch {ep}/{EPOCHS}: mse {hist[-1]:.5f} | {time.time() - t0_:.0f}s")
        torch.save(net.state_dict(), WORK / "aodnet_trained.pth")
        plt.figure(figsize=(5.2, 3.4)); plt.plot(range(1, EPOCHS + 1), hist, marker="o", ms=4, color=C_ORANGE); plt.xlabel("epoch"); plt.ylabel("train MSE"); plt.title("AOD-Net training"); plt.tight_layout(); savefig("fig_aod_training"); plt.show()
    net.eval()
    @torch.no_grad()
    def aod(hz):
        x = torch.tensor(hz, dtype=torch.float32).permute(2, 0, 1)[None].to(device)
        return np.clip(net(x)[0].permute(1, 2, 0).cpu().numpy().astype(np.float64), 0, 1)
else:
    ROUTE = "torch unavailable — AOD-Net skipped"
    aod = None


## 3. Evaluation on the test split — hazy input · CLAHE · DCP baseline · DCP tuned · AOD-Net
Same images, same metrics; paired Δ against the tuned DCP with 95 % CI and Wilcoxon p; runtime per image (DCP on CPU, AOD-Net on the device shown above).


In [ ]:
METHODS = ["hazy input", "CLAHE", "DCP baseline", "DCP tuned"] + (["AOD-Net"] if aod else [])
rows = []
for r in records:
    if r["split"] != "test": continue
    hz, g = get_pair(r)
    outs, ms = {}, {}
    t0_ = time.time(); outs["hazy input"] = hz; ms["hazy input"] = 0
    t0_ = time.time(); outs["CLAHE"] = clahe_enhance(hz); ms["CLAHE"] = (time.time() - t0_) * 1000
    t0_ = time.time(); outs["DCP baseline"] = dehaze(hz, **BASE)["dehazed"]; ms["DCP baseline"] = (time.time() - t0_) * 1000
    t0_ = time.time(); outs["DCP tuned"] = dehaze(hz, **tuned)["dehazed"]; ms["DCP tuned"] = (time.time() - t0_) * 1000
    if aod: t0_ = time.time(); outs["AOD-Net"] = aod(hz); ms["AOD-Net"] = (time.time() - t0_) * 1000
    for m in METHODS:
        rows.append({"name": r["name"], "dataset": r["dataset"], "stratum": r["stratum"], "real": r["real"], "method": m, **score_all(outs[m], g, hz), "ms": ms[m]})
E = pd.DataFrame(rows); E.to_csv(WORK / "dehaze_final_per_image.csv", index=False)
ref = E[E.method == "DCP tuned"].set_index("name"); out = []
for m in METHODS:
    sub = E[E.method == m].set_index("name").loc[ref.index]; mp, lo, hi = mean_ci(sub.psnr); ps = paired_stats(sub.psnr, ref.psnr)
    out.append({"method": m, "PSNR": mp, "PSNR lo": lo, "PSNR hi": hi, "SSIM": sub.ssim.mean(), "CIEDE": sub.ciede.mean(), "e": sub.e.mean(), "r": sub.r.mean(),
                "ΔPSNR vs DCP tuned": ps["mean"], "Δ lo": ps["lo"], "Δ hi": ps["hi"], "p": ps["p"], "ms/img": sub.ms.mean()})
final = pd.DataFrame(out).set_index("method")
table_png(final, "tab_final_comparison", f"Final comparison on the TEST split (n = {ref.shape[0]}) — AOD-Net route: {ROUTE}", highlight_max=("PSNR", "SSIM"), highlight_min=("CIEDE",))
by_ds = E.groupby(["dataset", "method"]).psnr.mean().unstack("method")[METHODS]; table_png(by_ds, "tab_final_by_dataset", "Test PSNR by dataset")
by_st = E.groupby(["stratum", "method"]).psnr.mean().unstack("method").reindex(STRATA)[METHODS]; table_png(by_st, "tab_final_by_stratum", "Test PSNR by density stratum")
fig, ax = plt.subplots(figsize=(7.2, 4))
x = np.arange(len(METHODS)); cols = [METHOD_COLORS[m] for m in METHODS]
ax.bar(x, final.PSNR, width=.62, color=cols, edgecolor=SURF)
ax.errorbar(x, final.PSNR, yerr=[final.PSNR - final["PSNR lo"], final["PSNR hi"] - final.PSNR], fmt="none", ecolor=INK2, capsize=3)
for xi, v, h_ in zip(x, final.PSNR, final["PSNR hi"]): ax.text(xi, h_ + 0.06, f"{v:.2f}", ha="center", va="bottom", fontsize=8.5, color=INK)
ax.set_xticks(x); ax.set_xticklabels(METHODS); ax.set_ylim(final["PSNR lo"].min() - 1.2, final["PSNR hi"].max() + 1.0); ax.set_ylabel("PSNR (dB), mean ± 95% CI")
ax.set_title("Classical vs learned on the same test images"); plt.tight_layout(); savefig("fig_final_comparison"); plt.show()


## 4. Density interaction between the learned and the classical model (H3)
Gap = PSNR(AOD-Net) − PSNR(DCP tuned) per image. H3 predicts the gap grows with density (learned model wins on dense haze; tuned DCP competitive on light/medium).


In [ ]:
if aod:
    gap = (E[E.method == "AOD-Net"].set_index("name").psnr - E[E.method == "DCP tuned"].set_index("name").psnr).rename("gap").reset_index().merge(S, on="name")
    rho_g, p_g = stats.spearmanr(gap.density, gap.gap)
    gt_ = gap.groupby("stratum").agg(n=("gap", "size"), gap_dB=("gap", "mean")).reindex(STRATA)
    for s in STRATA:
        m, lo, hi = mean_ci(gap[gap.stratum == s].gap); gt_.loc[s, "lo"] = lo; gt_.loc[s, "hi"] = hi
        gt_.loc[s, "AOD wins %"] = 100 * (gap[gap.stratum == s].gap > 0).mean()
    table_png(gt_, "tab_learned_gap_by_stratum", f"AOD-Net − DCP tuned by density stratum — Spearman ρ(density, gap) = {rho_g:+.2f} (p = {p_g:.2g})")
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    x = np.arange(3); ax[0].bar(x, gt_.gap_dB, width=.6, color=RAMP3, edgecolor=SURF)
    ax[0].errorbar(x, gt_.gap_dB, yerr=[gt_.gap_dB - gt_.lo, gt_.hi - gt_.gap_dB], fmt="none", ecolor=INK2, capsize=3); ax[0].axhline(0, color=RULE, lw=1)
    for xi, v, h_ in zip(x, gt_.gap_dB, gt_.hi): ax[0].text(xi, max(v, h_) + 0.03, f"{v:+.2f}", ha="center", va="bottom", fontsize=8.5, color=INK)
    ax[0].set_xticks(x); ax[0].set_xticklabels(STRATA); ax[0].set_ylabel("AOD-Net − DCP tuned (dB)"); ax[0].set_title("Learned-vs-classical gap by density (H3)")
    ax[1].scatter(gap.density, gap.gap, s=16, color=C_ORANGE, alpha=.7, edgecolor=SURF, linewidth=.4); ax[1].axhline(0, color=RULE, lw=1)
    ax[1].set_xlabel("haze density"); ax[1].set_ylabel("gap (dB)"); ax[1].set_title(f"per image   ρ = {rho_g:+.2f}")
    plt.tight_layout(); savefig("fig_learned_gap"); plt.show()
    GAP = dict(rho=float(rho_g), p=float(p_g), **{f"gap_{s}": float(gt_.loc[s, "gap_dB"]) for s in STRATA}, **{f"wins_{s}": float(gt_.loc[s, "AOD wins %"]) for s in STRATA})
else:
    GAP = {}


## 5. Qualitative comparison on the same test images


In [ ]:
by_name = {r["name"]: r for r in records}
order = ref.psnr.sort_values().index
picks = [order[0], order[len(order) // 3], order[2 * len(order) // 3], order[-1]]
fig, ax = plt.subplots(len(picks), len(METHODS) + 1, figsize=(2.55 * (len(METHODS) + 1), 2.6 * len(picks)))
for i, nm in enumerate(picks):
    r = by_name[nm]; hz, g = get_pair(r)
    outs = {"hazy input": hz, "CLAHE": clahe_enhance(hz), "DCP baseline": dehaze(hz, **BASE)["dehazed"], "DCP tuned": dehaze(hz, **tuned)["dehazed"]}
    if aod: outs["AOD-Net"] = aod(hz)
    for j, m in enumerate(METHODS):
        ax[i, j].imshow(outs[m], vmin=0, vmax=1); ax[i, j].set_title(f"{m}  {psnr(outs[m], g):.1f} dB", fontsize=7.5); ax[i, j].axis("off")
    ax[i, -1].imshow(g, vmin=0, vmax=1); ax[i, -1].set_title("ground truth", fontsize=7.5); ax[i, -1].axis("off")
    ax[i, 0].text(0.01, 0.98, f"{r['dataset']} · {r['stratum']}", transform=ax[i, 0].transAxes, fontsize=7, va="top", color="w", bbox=dict(facecolor=INK, alpha=.55, pad=2, lw=0))
fig.suptitle("Same test images — every method (rows: worst → best for the tuned DCP)", y=1.0, fontweight="bold"); plt.tight_layout(); savefig("fig_qualitative_methods"); plt.show()


## 6. Save results + LaTeX macros


In [ ]:
res = {"source": SOURCE, "fingerprint": FP, "aod_route": ROUTE, "methods": METHODS, "final_test": final.round(4).to_dict("index"),
       "by_dataset": by_ds.round(3).to_dict("index"), "by_stratum": by_st.round(3).to_dict("index"), "gap": GAP, "tuned_cfg": tuned}
json.dump(res, open(WORK / "dehaze_learned_results.json", "w"), indent=1, default=float)
fmt2 = lambda v: f"{v:.2f}"; fmt3 = lambda v: f"{v:.3f}"
def fmtp(p):
    """p-value for LaTeX math mode: 0.023, or 3.8 x 10^-5 (TeX times) below 1e-3."""
    if p != p: return r"\mathrm{n/a}"
    if p >= 1e-3: return f"{p:.3f}"
    m, e = f"{p:.1e}".split("e"); return m + "\\times10^{" + str(int(e)) + "}"
macros = {"aodRoute": ROUTE.replace("_", r"\_"), "nTestLearned": int(ref.shape[0])}
for m in METHODS:
    key = {"hazy input": "Input", "CLAHE": "Clahe", "DCP baseline": "Base", "DCP tuned": "Tuned", "AOD-Net": "Aod"}[m]
    macros.update({f"fin{key}PSNR": fmt2(final.loc[m, "PSNR"]), f"fin{key}SSIM": fmt3(final.loc[m, "SSIM"]), f"fin{key}CIEDE": fmt2(final.loc[m, "CIEDE"]),
                   f"fin{key}E": fmt2(final.loc[m, "e"]), f"fin{key}R": fmt2(final.loc[m, "r"]), f"fin{key}Ms": f"{final.loc[m, 'ms/img']:.0f}",
                   f"fin{key}Delta": fmt2(final.loc[m, "ΔPSNR vs DCP tuned"]), f"fin{key}Lo": fmt2(final.loc[m, "Δ lo"]), f"fin{key}Hi": fmt2(final.loc[m, "Δ hi"]), f"fin{key}P": fmtp(final.loc[m, 'p'])})
if GAP:
    macros.update({"gapRho": f"{GAP['rho']:+.2f}", "gapP": fmtp(GAP['p']), "gapLight": fmt2(GAP["gap_light"]), "gapMedium": fmt2(GAP["gap_medium"]), "gapDense": fmt2(GAP["gap_dense"]),
                   "winsLight": f"{GAP['wins_light']:.0f}", "winsMedium": f"{GAP['wins_medium']:.0f}", "winsDense": f"{GAP['wins_dense']:.0f}"})
tex_macros(macros, WORK / "numbers_learned.tex")
print(json.dumps({"aod_route": ROUTE, "final_test_psnr": {m: round(final.loc[m, "PSNR"], 2) for m in METHODS}, "gap": GAP}, indent=1))


## Summary
The learned reference is evaluated under exactly the protocol of NB 2 (same test images, metrics and paired statistics), and the
route by which its weights were obtained is recorded. Together with NB 2 this closes the plan: stage attribution (H1),
bright-region failure (H2), density interaction for DCP alone and against a learned model (H3).
